In [6]:
pip install pandas networkx scikit-learn numpy

In [16]:
import json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.ensemble import IsolationForest

DUST_XRP = 1.0          # payments below this are treated as dust
QUICK_FORWARD_MIN = 10  # "rapid" = first send within this many minutes of first receipt

raw = pd.read_parquet("transactions.parquet")
print(f"Loaded {len(raw)} rows")

# Step 0: normalise the raw XRPL schema into src / dst / xrp / tag / time.
raw = raw[(raw.TRANSACTION_TYPE == "Payment") & (raw.SUCCESS == True)].copy()

def parse_json(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, (dict, str)) and not isinstance(x, dict):
        try:
            return json.loads(x)
        except (ValueError, TypeError):
            return None
    return x

# Delivered amount: a JSON-encoded string of drops for XRP, an object for issued currencies.
def to_xrp(x):
    v = parse_json(x)
    if isinstance(v, (str, int)):          # native XRP, in drops
        try:
            return int(v) / 1e6
        except ValueError:
            return np.nan
    return np.nan                           # IOU (dict) or missing: not XRP

# Destination / tag: use dedicated columns if present, else dig them out of the extras blob.
if "DESTINATION" in raw.columns:
    dst = raw["DESTINATION"]
    tag = raw["DESTINATION_TAG"] if "DESTINATION_TAG" in raw.columns else pd.Series(np.nan, index=raw.index)
else:
    extra = raw["EXTRA_TRANSACTION_FIELDS"].map(lambda x: parse_json(x) or {})
    dst = extra.map(lambda d: d.get("Destination"))
    tag = extra.map(lambda d: d.get("DestinationTag"))

df = pd.DataFrame({
    "time": pd.to_datetime(raw["DATETIME"], utc=True),
    "src": raw["ACCOUNT"],
    "dst": dst,
    "xrp": raw["DELIVERED_AMOUNT"].map(to_xrp),
    "tag": pd.to_numeric(tag, errors="coerce"),
})
n_iou = df.xrp.isna().sum()
df = df.dropna(subset=["xrp", "dst"])
df = df[df.src != df.dst]                   # self-payments carry no flow information
print(f"Loaded {len(df)} XRP payments ({n_iou} non-XRP / unparseable rows excluded)")

# Step 1: separate dust from value. Dust spam dominates raw XRPL graphs and drowns the signal.
dust = df[df.xrp < DUST_XRP]
value = df[df.xrp >= DUST_XRP].copy()
print(f"{len(dust)} dust payments set aside, {len(value)} value payments kept")

# Step 2: per-account behavioural features. Each one maps to a typology in this module.
out = value.groupby("src").agg(out_n=("xrp", "size"), out_xrp=("xrp", "sum"),
                               out_cpty=("dst", "nunique"), first_out=("time", "min"),
                               out_size_cv=("xrp", lambda s: s.std() / s.mean() if len(s) > 1 else np.nan))
inc = value.groupby("dst").agg(in_n=("xrp", "size"), in_xrp=("xrp", "sum"),
                               in_cpty=("src", "nunique"), first_in=("time", "min"),
                               in_tags=("tag", "nunique"))
feat = out.join(inc, how="outer")
feat[["out_n", "out_xrp", "out_cpty", "in_n", "in_xrp", "in_cpty", "in_tags"]] = \
    feat[["out_n", "out_xrp", "out_cpty", "in_n", "in_xrp", "in_cpty", "in_tags"]].fillna(0)

# Pass-through: how much of what came in went straight back out (mule / layering hop).
both = (feat.in_xrp > 0) & (feat.out_xrp > 0)
feat["pass_through"] = 0.0
feat.loc[both, "pass_through"] = (feat.loc[both, ["in_xrp", "out_xrp"]].min(axis=1)
                                  / feat.loc[both, ["in_xrp", "out_xrp"]].max(axis=1))
# Minutes between first receipt and first send (rapid movement, FATF red flag).
feat["minutes_to_forward"] = (feat.first_out - feat.first_in).dt.total_seconds() / 60
# Fan-out / fan-in: many distinct counterparties (smurfing out, consolidation in).
feat["fan_out"] = feat.out_cpty
feat["fan_in"] = feat.in_cpty
# Structuring: many outgoing payments of near-identical size (low coefficient of variation).
feat["uniform_splits"] = (feat.out_n >= 5) & (feat.out_size_cv.fillna(1) < 0.05)

# Step 3: transparent rule-based flags. Judges and investigators can read these.
feat["flag_rapid_pass_through"] = (feat.pass_through >= 0.9) & feat.minutes_to_forward.between(0, QUICK_FORWARD_MIN)
feat["flag_fan_out"] = feat.fan_out >= 10
feat["flag_fan_in"] = feat.fan_in >= 10
feat["flag_structuring"] = feat.uniform_splits
flags = [c for c in feat.columns if c.startswith("flag_")]
feat["rule_score"] = feat[flags].sum(axis=1)

# Step 4: cycles. Value that returns to where it started within a few hops (round-tripping).
# Note: length_bound requires networkx >= 3.1.
G = nx.DiGraph()
edges = value.groupby(["src", "dst"]).xrp.sum().reset_index()
G.add_weighted_edges_from(edges.itertuples(index=False, name=None), weight="xrp")
cycles = [c for c in nx.simple_cycles(G, length_bound=4) if len(c) > 1]
in_cycle = {a for c in cycles for a in c}
feat["in_short_cycle"] = feat.index.isin(in_cycle)
print(f"{len(cycles)} short cycles (2 to 4 hops) found")

# Step 5: an unsupervised baseline. No labels, so we rank by "unusualness", not guilt.
cols = ["out_n", "out_xrp", "out_cpty", "in_n", "in_xrp", "in_cpty", "pass_through"]
X = np.log1p(feat[cols].clip(lower=0))
model = IsolationForest(n_estimators=300, contamination="auto", random_state=42).fit(X)
feat["anomaly_score"] = -model.score_samples(X)          # higher means more unusual

ranked = feat.sort_values(["rule_score", "anomaly_score"], ascending=False)
ranked.to_csv("account_features.csv")
print("\nFlag counts:")
print(feat[flags + ["in_short_cycle"]].sum().to_string())
print("\nTop 10 accounts to review (a review queue, not an accusation):")
print(ranked[["rule_score", "anomaly_score", "in_xrp", "out_xrp", "fan_in", "fan_out",
              "pass_through", "minutes_to_forward"]].head(10).round(3).to_string())

Loaded 40101 rows
Loaded 14598 XRP payments (25503 non-XRP / unparseable rows excluded)
16 dust payments set aside, 14582 value payments kept
8182 short cycles (2 to 4 hops) found

Flag counts:
flag_rapid_pass_through      1
flag_fan_out               465
flag_fan_in                424
flag_structuring             0
in_short_cycle             674

Top 10 accounts to review (a review queue, not an accusation):
                                    rule_score  anomaly_score     in_xrp    out_xrp  fan_in  fan_out  pass_through  minutes_to_forward
r3a9MLXwAtnXvRxYQP8tu3XPp3Tgm12vB7           3          0.430   7809.443   7425.336      17     21.0         0.951               8.000
rNqJ9981mBL8Q1kLawD1q1CF6KJ7gntnnY           2          0.629   4218.925   3834.735      44     60.0         0.909              28.467
rNMg8RxqcN7LsvvSgaDjWPMb6njcartPPs           2          0.607   1101.030   3891.990      24     61.0         0.283            -195.200
rGMS5qVqVmTbqpdBTZPLSxdeobBbz3VTLj           2 

In [17]:
import json
import re
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
DUST_XRP = 1.0
QUICK_FORWARD_MIN = 10
MAX_CYCLE_HOPS = 4
MAX_CYCLE_MINUTES = 60

# Stop rather than silently report incomplete cycle-search results.
MAX_SEARCH_STATES = 2_000_000
MAX_CYCLES = 100_000


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def parse_json_object(value):
    """Accept JSON strings or already-decoded dictionaries."""
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            decoded = json.loads(value)
            return decoded if isinstance(decoded, dict) else {}
        except (json.JSONDecodeError, TypeError):
            pass

    return {}


def parse_xrp_drops(value):
    """
    Parse native XRP amounts as integer drops.

    Accepts integer drops or JSON-encoded integer strings.
    Rejects token objects, unavailable amounts and malformed values.
    """
    if isinstance(value, str):
        value = value.strip()
        try:
            value = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            pass

    if isinstance(value, bool):
        return None

    if isinstance(value, (int, np.integer)):
        drops = int(value)
    elif isinstance(value, str) and re.fullmatch(r"\d+", value):
        drops = int(value)
    else:
        return None

    return drops if drops >= 0 else None


# ---------------------------------------------------------------------
# Load and normalise the parquet data
# ---------------------------------------------------------------------
raw = pd.read_parquet("transactions.parquet")
print(f"Loaded {len(raw):,} transactions")

required = {
    "TRANSACTION_TYPE",
    "TRANSACTION_RESULT",
    "DATETIME",
    "DELIVERED_AMOUNT",
}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

payments = raw.loc[
    raw["TRANSACTION_TYPE"].eq("Payment")
    & raw["TRANSACTION_RESULT"].eq("tesSUCCESS")
].copy()

if "TRANSACTION_HASH" in payments.columns:
    duplicate = (
        payments["TRANSACTION_HASH"].notna()
        & payments["TRANSACTION_HASH"].duplicated()
    )
    print(f"{duplicate.sum():,} duplicate transaction hashes excluded")
    payments = payments.loc[~duplicate].copy()

payments = payments.reset_index(drop=True)

extra = (
    payments["EXTRA_TRANSACTION_FIELDS"].map(parse_json_object)
    if "EXTRA_TRANSACTION_FIELDS" in payments.columns
    else pd.Series([{} for _ in range(len(payments))], index=payments.index)
)


def get_field(column, json_key):
    """Prefer a dedicated column, falling back to transaction JSON."""
    fallback = extra.map(lambda obj: obj.get(json_key))
    if column in payments.columns:
        return payments[column].combine_first(fallback)
    return fallback


df = pd.DataFrame(index=payments.index)
df["src"] = get_field("ACCOUNT", "Account")
df["dst"] = get_field("DESTINATION", "Destination")
df["tag"] = get_field("DESTINATION_TAG", "DestinationTag")
df["time"] = pd.to_datetime(
    payments["DATETIME"], utc=True, errors="coerce"
)

# Nullable integer drops preserve exact amounts for FIFO matching.
df["drops"] = pd.array(
    [parse_xrp_drops(v) for v in payments["DELIVERED_AMOUNT"]],
    dtype="Int64",
)

if "TRANSACTION_HASH" in payments.columns:
    df["transaction_hash"] = payments["TRANSACTION_HASH"]
else:
    df["transaction_hash"] = pd.NA

# A payment delivering XRP may have spent tokens.
# Exclude transactions with non-XRP or unparseable SendMax values.
send_max = get_field("SEND_MAX", "SendMax")
parsed_send_max = send_max.map(parse_xrp_drops)
cross_currency = send_max.notna() & parsed_send_max.isna()

valid_addresses = (
    df["src"].map(lambda v: isinstance(v, str) and bool(v.strip()))
    & df["dst"].map(lambda v: isinstance(v, str) and bool(v.strip()))
)
valid = (
    valid_addresses
    & df["time"].notna()
    & df["drops"].gt(0).fillna(False)
    & ~cross_currency
)

print(
    f"{(~valid).sum():,} token/cross-currency payments or "
    "rows with invalid/missing fields excluded"
)

df = df.loc[valid].copy()
df["drops"] = df["drops"].astype("int64")
df["xrp"] = df["drops"] / 1_000_000

# Self-payments do not represent movement between distinct accounts.
self_payment = df["src"].eq(df["dst"])
print(f"{self_payment.sum():,} self-payments excluded")
df = df.loc[~self_payment].copy()

print(f"{len(df):,} usable XRP payments")


# ---------------------------------------------------------------------
# Step 1: Separate dust from value payments
# ---------------------------------------------------------------------
dust = df.loc[df["xrp"] < DUST_XRP].copy()
value = df.loc[df["xrp"] >= DUST_XRP].copy()

print(
    f"{len(dust):,} dust payments set aside, "
    f"{len(value):,} value payments kept"
)

if value.empty:
    raise ValueError(
        "No value payments remain. Check the data and DUST_XRP."
    )

# IDs are local to this analysis; transaction hashes are retained for audit.
tx = value.reset_index(drop=True).copy()
tx["transfer_id"] = np.arange(len(tx))


# ---------------------------------------------------------------------
# Step 2: Per-account behavioural features
# ---------------------------------------------------------------------
out = tx.groupby("src").agg(
    out_n=("xrp", "size"),
    out_xrp=("xrp", "sum"),
    out_cpty=("dst", "nunique"),
    first_out=("time", "min"),
    out_size_cv=(
        "xrp",
        lambda s: s.std() / s.mean() if len(s) > 1 else np.nan,
    ),
)

inc = tx.groupby("dst").agg(
    in_n=("xrp", "size"),
    in_xrp=("xrp", "sum"),
    in_cpty=("src", "nunique"),
    first_in=("time", "min"),
    in_tags=("tag", "nunique"),
)

feat = out.join(inc, how="outer")
feat.index.name = "account"

count_amount_cols = [
    "out_n", "out_xrp", "out_cpty",
    "in_n", "in_xrp", "in_cpty", "in_tags",
]
feat[count_amount_cols] = feat[count_amount_cols].fillna(0)

feat["fan_out"] = feat["out_cpty"]
feat["fan_in"] = feat["in_cpty"]
feat["uniform_splits"] = (
    (feat["out_n"] >= 5)
    & (feat["out_size_cv"].fillna(1) < 0.05)
)


# ---------------------------------------------------------------------
# Step 3: Match individual receipts to subsequent sends using FIFO
# ---------------------------------------------------------------------
# Matching is an allocation assumption, not proof of fund provenance.
# Only retained value payments are used.
#
# Equal timestamps are handled conservatively: outgoing events are
# processed before incoming events, preventing same-time receipt funding.
events = []

for row in tx.itertuples(index=False):
    events.append(
        (row.time, 0, row.transfer_id, row.src, int(row.drops))
    )
    events.append(
        (row.time, 1, row.transfer_id, row.dst, int(row.drops))
    )

events.sort(key=lambda event: (event[0], event[1], event[2]))

receipts = defaultdict(deque)
match_rows = []

for timestamp, event_type, transfer_id, account, drops in events:
    if event_type == 1:
        receipts[account].append(
            {
                "transfer_id": transfer_id,
                "time": timestamp,
                "remaining": drops,
            }
        )
        continue

    remaining_send = drops
    queue = receipts[account]

    while remaining_send > 0 and queue:
        receipt = queue[0]
        allocated = min(remaining_send, receipt["remaining"])

        match_rows.append(
            {
                "account": account,
                "receipt_id": receipt["transfer_id"],
                "send_id": transfer_id,
                "receipt_time": receipt["time"],
                "send_time": timestamp,
                "matched_drops": allocated,
                "minutes_to_forward": (
                    timestamp - receipt["time"]
                ).total_seconds() / 60,
            }
        )

        remaining_send -= allocated
        receipt["remaining"] -= allocated

        if receipt["remaining"] == 0:
            queue.popleft()

    # Any remaining send amount has no earlier receipt available in this
    # dataset. It may reflect an opening balance or excluded transactions.

matched = pd.DataFrame(
    match_rows,
    columns=[
        "account", "receipt_id", "send_id",
        "receipt_time", "send_time",
        "matched_drops", "minutes_to_forward",
    ],
)
matched["matched_xrp"] = matched["matched_drops"] / 1_000_000

feat["matched_out_xrp"] = 0.0
feat["rapid_matched_xrp"] = 0.0
feat["minutes_to_forward"] = np.nan

if not matched.empty:
    total_matched = matched.groupby("account")["matched_xrp"].sum()

    rapid = matched.loc[
        matched["minutes_to_forward"].between(0, QUICK_FORWARD_MIN)
    ]
    rapid_matched = rapid.groupby("account")["matched_xrp"].sum()

    weighted = matched.assign(
        weighted_minutes=(
            matched["matched_xrp"] * matched["minutes_to_forward"]
        )
    ).groupby("account").agg(
        weighted_minutes=("weighted_minutes", "sum"),
        matched_xrp=("matched_xrp", "sum"),
    )

    feat["matched_out_xrp"] = total_matched.reindex(
        feat.index, fill_value=0
    )
    feat["rapid_matched_xrp"] = rapid_matched.reindex(
        feat.index, fill_value=0
    )

    # Amount-weighted mean delay for matched portions only.
    feat["minutes_to_forward"] = (
        weighted["weighted_minutes"] / weighted["matched_xrp"]
    ).reindex(feat.index)

# Fraction of outgoing value allocated to earlier observed receipts.
feat["pass_through"] = (
    feat["matched_out_xrp"]
    .div(feat["out_xrp"].replace(0, np.nan))
    .fillna(0)
    .clip(0, 1)
)

# Fraction of outgoing value allocated to sufficiently recent receipts.
feat["rapid_pass_through"] = (
    feat["rapid_matched_xrp"]
    .div(feat["out_xrp"].replace(0, np.nan))
    .fillna(0)
    .clip(0, 1)
)

feat["unmatched_out_xrp"] = (
    feat["out_xrp"] - feat["matched_out_xrp"]
).clip(lower=0)


# ---------------------------------------------------------------------
# Step 4: Transparent rule-based flags
# ---------------------------------------------------------------------
feat["flag_rapid_pass_through"] = (
    feat["rapid_pass_through"] >= 0.9
)
feat["flag_fan_out"] = feat["fan_out"] >= 10
feat["flag_fan_in"] = feat["fan_in"] >= 10
feat["flag_structuring"] = feat["uniform_splits"]

flags = [column for column in feat.columns if column.startswith("flag_")]
feat["rule_score"] = feat[flags].sum(axis=1)


# ---------------------------------------------------------------------
# Step 5: Chronological cycles linked by FIFO allocations
# ---------------------------------------------------------------------
# Each consecutive transfer must:
#   1. Start at the destination of the previous transfer.
#   2. Occur strictly later.
#   3. Have a positive FIFO allocation from the previous transfer.
#
# Routes must return to their starting account within the configured
# hop/time limits, without repeating intermediate accounts.
transfers = tx.set_index("transfer_id")[
    ["src", "dst", "time", "drops"]
].to_dict("index")

successors = defaultdict(list)
for row in matched.itertuples(index=False):
    successors[int(row.receipt_id)].append(
        (int(row.send_id), int(row.matched_drops))
    )


def find_temporal_cycles(transfers, successors):
    cycle_rows = []
    search_states = 0

    def walk(start_id, path, accounts, link_capacity):
        nonlocal search_states

        search_states += 1
        if search_states > MAX_SEARCH_STATES:
            raise RuntimeError(
                "Cycle search exceeded MAX_SEARCH_STATES. "
                "Reduce the hop/time limits or analyse smaller periods. "
                "No partial cycle results have been exported."
            )

        if len(path) >= MAX_CYCLE_HOPS:
            return

        first = transfers[start_id]
        current = transfers[path[-1]]

        for next_id, allocated_drops in successors.get(path[-1], []):
            nxt = transfers[next_id]

            if nxt["src"] != current["dst"]:
                continue
            if nxt["time"] <= current["time"]:
                continue

            duration = (
                nxt["time"] - first["time"]
            ).total_seconds() / 60

            if duration > MAX_CYCLE_MINUTES:
                continue

            next_path = path + [next_id]
            next_capacity = min(link_capacity, allocated_drops)

            if nxt["dst"] == first["src"]:
                if len(cycle_rows) >= MAX_CYCLES:
                    raise RuntimeError(
                        "Cycle search exceeded MAX_CYCLES. "
                        "Reduce the hop/time limits or analyse smaller periods. "
                        "No partial cycle results have been exported."
                    )

                cycle_rows.append(
                    {
                        "accounts": " -> ".join(
                            accounts + [nxt["dst"]]
                        ),
                        "transfer_ids": json.dumps(
                            [int(i) for i in next_path]
                        ),
                        "hops": len(next_path),
                        "start_time": first["time"],
                        "end_time": nxt["time"],
                        "duration_minutes": duration,
                        # Pairwise allocations do not prove that the same
                        # portion of funds traversed every hop.
                        "fifo_link_upper_bound_xrp": (
                            next_capacity / 1_000_000
                        ),
                    }
                )
                continue

            if nxt["dst"] in accounts:
                continue

            walk(
                start_id,
                next_path,
                accounts + [nxt["dst"]],
                next_capacity,
            )

    for start_id, transfer in transfers.items():
        # A transfer without a successor cannot begin a linked cycle.
        if start_id not in successors:
            continue

        walk(
            start_id=start_id,
            path=[start_id],
            accounts=[transfer["src"], transfer["dst"]],
            link_capacity=int(transfer["drops"]),
        )

    return pd.DataFrame(
        cycle_rows,
        columns=[
            "accounts", "transfer_ids", "hops",
            "start_time", "end_time", "duration_minutes",
            "fifo_link_upper_bound_xrp",
        ],
    )


temporal_cycles = find_temporal_cycles(transfers, successors)

in_cycle = set()
for ids_json in temporal_cycles["transfer_ids"]:
    for transfer_id in json.loads(ids_json):
        transfer = transfers[transfer_id]
        in_cycle.update([transfer["src"], transfer["dst"]])

feat["in_short_cycle"] = feat.index.isin(in_cycle)

print(
    f"{len(temporal_cycles):,} chronological FIFO-linked cycles "
    f"(2–{MAX_CYCLE_HOPS} hops) found"
)


# ---------------------------------------------------------------------
# Step 6: Unsupervised anomaly baseline
# ---------------------------------------------------------------------
# Scores indicate unusualness within this dataset, not guilt.
model_cols = [
    "out_n", "out_xrp", "out_cpty",
    "in_n", "in_xrp", "in_cpty",
    "pass_through", "rapid_pass_through",
]
X = np.log1p(feat[model_cols].clip(lower=0))

if len(feat) >= 2:
    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=42,
    )
    model.fit(X)
    feat["anomaly_score"] = -model.score_samples(X)
else:
    feat["anomaly_score"] = np.nan
    print("Too few accounts for meaningful anomaly ranking.")

ranked = feat.sort_values(
    ["rule_score", "anomaly_score"],
    ascending=False,
)


# ---------------------------------------------------------------------
# Step 7: Export audit tables and review queue
# ---------------------------------------------------------------------
ranked.to_csv("account_features.csv")
tx.to_csv("transfers_with_ids.csv", index=False)
matched.to_csv("matched_transfers.csv", index=False)
temporal_cycles.to_csv("temporal_cycles.csv", index=False)

print("\nFlag counts:")
print(feat[flags + ["in_short_cycle"]].sum().to_string())

print("\nTop 10 accounts to review (a review queue, not an accusation):")
print(
    ranked[
        [
            "rule_score",
            "anomaly_score",
            "in_xrp",
            "out_xrp",
            "fan_in",
            "fan_out",
            "pass_through",
            "rapid_pass_through",
            "minutes_to_forward",
            "unmatched_out_xrp",
            "in_short_cycle",
        ]
    ].head(10).round(3).to_string()
)

print(
    "\nInterpretation: FIFO links are inferred allocations. "
    "Opening balances, excluded payments and the observation window "
    "affect matching. Equal-timestamp transfers are not linked. "
    "Cycle amounts are upper bounds; overlapping cycles must not be summed."
)

Loaded 40,101 transactions
0 duplicate transaction hashes excluded
29,544 token/cross-currency payments or rows with invalid/missing fields excluded
0 self-payments excluded
10,557 usable XRP payments
13 dust payments set aside, 10,544 value payments kept
3 chronological FIFO-linked cycles (2–4 hops) found

Flag counts:
flag_rapid_pass_through      0
flag_fan_out               264
flag_fan_in                264
flag_structuring             0
in_short_cycle               6

Top 10 accounts to review (a review queue, not an accusation):
                                    rule_score  anomaly_score    in_xrp    out_xrp  fan_in  fan_out  pass_through  rapid_pass_through  minutes_to_forward  unmatched_out_xrp  in_short_cycle
account                                                                                                                                                                                     
rGMS5qVqVmTbqpdBTZPLSxdeobBbz3VTLj           2          0.624  1486.118   2917.25

In [19]:
import json
from collections import defaultdict, deque

import numpy as np
import pandas as pd


HOUR = 3600
TEN_MINUTES = 600
DAY = 24 * HOUR

HORIZONS = {
    "1h": HOUR,
    "1d": DAY,
    "7d": 7 * DAY,
}

REQUIRED_COLUMNS = {
    "TRANSACTION_HASH",
    "TIMESTAMP",
    "FROM_ADDRESS",
    "TO_ADDRESS",
    "VALUE",
}


def as_unix_seconds(value):
    """Accept Unix seconds or a datetime-like boundary."""
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")

    return timestamp.timestamp()


def prepare_transactions(df):
    """
    Input assumptions:
      - One row per successful native-XRP payment.
      - VALUE is the actual delivered amount in XRP, not drops.
      - Cross-currency payments have already been excluded.
      - TIMESTAMP contains Unix seconds.
      - Addresses and transaction hashes are unmodified identifiers.

    Raw XRPL exports must be normalised before calling this function.
    """
    missing = REQUIRED_COLUMNS.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    data = df.copy()

    identifier_columns = [
        "TRANSACTION_HASH", "FROM_ADDRESS", "TO_ADDRESS"
    ]
    for column in identifier_columns:
        valid = data[column].map(
            lambda value: isinstance(value, str) and bool(value.strip())
        )
        if not valid.all():
            raise ValueError(f"{column} contains missing/invalid identifiers.")

    data["TIMESTAMP"] = pd.to_numeric(
        data["TIMESTAMP"], errors="raise"
    )
    data["VALUE"] = pd.to_numeric(data["VALUE"], errors="raise")

    numeric = data[["TIMESTAMP", "VALUE"]].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("TIMESTAMP and VALUE must contain finite numbers.")

    if (data["VALUE"] <= 0).any():
        raise ValueError("VALUE must be positive.")

    # Fail on conflicting duplicates rather than arbitrarily choosing a row.
    compare_columns = [
        "TIMESTAMP", "FROM_ADDRESS", "TO_ADDRESS", "VALUE"
    ]
    conflicts = (
        data.groupby("TRANSACTION_HASH")[compare_columns]
        .nunique(dropna=False)
        .gt(1)
        .any(axis=1)
    )
    if conflicts.any():
        raise ValueError("Conflicting rows share a TRANSACTION_HASH.")

    duplicate_count = int(data["TRANSACTION_HASH"].duplicated().sum())
    data = data.drop_duplicates("TRANSACTION_HASH").copy()

    # Native XRP has integer-drop precision.
    drop_values = data["VALUE"].to_numpy(dtype=float) * 1_000_000
    rounded = np.rint(drop_values)

    if not np.allclose(drop_values, rounded, rtol=0, atol=0.01):
        raise ValueError(
            "VALUE must be XRP with integer-drop precision, not token values."
        )

    # XRPL supply is below this bound; it also protects integer conversion.
    if (rounded > 100_000_000_000_000_000).any():
        raise ValueError("Implausible native-XRP amount; check VALUE units.")

    data["DROPS"] = rounded.astype("int64")
    if (data["DROPS"] <= 0).any():
        raise ValueError("A payment is smaller than one drop.")

    self_payment = data["FROM_ADDRESS"].eq(data["TO_ADDRESS"])
    self_payment_count = int(self_payment.sum())

    data = (
        data.loc[~self_payment]
        .sort_values(
            ["TIMESTAMP", "TRANSACTION_HASH"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    data["transfer_id"] = np.arange(len(data))

    return data, {
        "duplicate_rows_removed": duplicate_count,
        "self_payments_removed": self_payment_count,
    }


def prepare_services(services):
    """
    Registry columns:
      address, service_name, service_type, source, verified_date

    Registry entries must be independently verified.
    For historical evaluation, use a registry snapshot available at the
    evaluation cutoff; a newly discovered label must not leak backwards.
    """
    columns = [
        "address", "service_name", "service_type", "source", "verified_date"
    ]

    if services is None or services.empty:
        return pd.DataFrame(columns=columns).set_index("address")

    missing = set(columns).difference(services.columns)
    if missing:
        raise ValueError(
            f"Service registry is missing: {sorted(missing)}"
        )

    registry = services[columns].copy()

    for column in columns:
        valid = registry[column].map(
            lambda value: isinstance(value, str) and bool(value.strip())
        )
        if not valid.all():
            raise ValueError(
                f"Service registry field {column!r} is missing/invalid."
            )

    if registry["address"].duplicated().any():
        raise ValueError("Service registry addresses must be unique.")

    registry["verified_date"] = pd.to_datetime(
        registry["verified_date"], utc=True, errors="raise"
    )

    return registry.set_index("address")


def fifo_match(data):
    """
    Allocate each send against the account's earlier observed receipts.

    Includes dust and known services. Allocation occurs before any
    reporting/model exclusions.

    Equal-timestamp receipts cannot fund equal-timestamp sends.
    Opening balances and transactions outside the dataset are unknown.
    """
    events = []

    for row in data.itertuples(index=False):
        events.append(
            (
                row.TIMESTAMP, 0, row.transfer_id,
                row.FROM_ADDRESS, int(row.DROPS),
            )
        )
        events.append(
            (
                row.TIMESTAMP, 1, row.transfer_id,
                row.TO_ADDRESS, int(row.DROPS),
            )
        )

    events.sort(key=lambda event: (event[0], event[1], event[2]))

    available = defaultdict(deque)
    matches = []

    for timestamp, event_type, transfer_id, account, drops in events:
        if event_type == 1:
            available[account].append(
                [transfer_id, timestamp, drops]
            )
            continue

        remaining = drops
        queue = available[account]

        while remaining > 0 and queue:
            receipt_id, receipt_time, receipt_remaining = queue[0]
            allocated = min(remaining, receipt_remaining)

            matches.append(
                {
                    "account": account,
                    "receipt_id": receipt_id,
                    "send_id": transfer_id,
                    "receipt_timestamp": receipt_time,
                    "send_timestamp": timestamp,
                    "matched_drops": allocated,
                    "delay_seconds": timestamp - receipt_time,
                }
            )

            remaining -= allocated
            queue[0][2] -= allocated

            if queue[0][2] == 0:
                queue.popleft()

    result = pd.DataFrame(
        matches,
        columns=[
            "account", "receipt_id", "send_id",
            "receipt_timestamp", "send_timestamp",
            "matched_drops", "delay_seconds",
        ],
    )
    result["matched_xrp"] = result["matched_drops"] / 1_000_000

    hashes = data.set_index("transfer_id")["TRANSACTION_HASH"]
    result["receipt_hash"] = result["receipt_id"].map(hashes)
    result["send_hash"] = result["send_id"].map(hashes)

    return result


def make_features(
    df,
    *,
    services=None,
    dust_xrp=1.0,
    horizons=None,
    mode="causal",
    report_start=None,
    report_end=None,
    observed_start=None,
    observed_end=None,
):
    """
    Return features, metadata and evidence as separate tables.

    mode="causal":
        Features use past events and the current transaction only.

    mode="retrospective":
        Also includes later FIFO-linked sends. Appropriate for delayed
        review, not a model claiming detection at transaction time.

    Reporting interval: [report_start, report_end).
    Observation interval: [observed_start, observed_end).

    Supply observation boundaries from ingestion coverage when possible.
    The code cannot detect missing transactions or gaps within that interval.

    Counts/amounts use non-dust payments. FIFO uses all supplied payments.
    """
    if mode not in {"causal", "retrospective"}:
        raise ValueError("mode must be 'causal' or 'retrospective'.")

    if not np.isfinite(dust_xrp) or dust_xrp < 0:
        raise ValueError("dust_xrp must be finite and non-negative.")

    horizons = HORIZONS.copy() if horizons is None else dict(horizons)
    if not horizons or any(
        not np.isfinite(seconds) or seconds <= 0
        for seconds in horizons.values()
    ):
        raise ValueError("Provide positive, finite horizon lengths.")

    data, audit = prepare_transactions(df)
    if data.empty:
        raise ValueError("No usable transactions remain.")

    registry = prepare_services(services)

    start = (
        float(data["TIMESTAMP"].min())
        if observed_start is None
        else as_unix_seconds(observed_start)
    )
    end = (
        float(data["TIMESTAMP"].max()) + 1
        if observed_end is None
        else as_unix_seconds(observed_end)
    )

    if start >= end:
        raise ValueError("observed_start must precede observed_end.")

    if (
        (data["TIMESTAMP"] < start)
        | (data["TIMESTAMP"] >= end)
    ).any():
        raise ValueError("Transactions fall outside observation boundaries.")

    reporting_start = (
        start if report_start is None else as_unix_seconds(report_start)
    )
    reporting_end = (
        end if report_end is None else as_unix_seconds(report_end)
    )

    if not start <= reporting_start < reporting_end <= end:
        raise ValueError(
            "Reporting interval must be inside the observation interval."
        )

    data["is_dust"] = data["VALUE"] < dust_xrp
    matches = fifo_match(data)

    # Activity features exclude dust; FIFO evidence does not.
    value = data.loc[~data["is_dust"]].copy()

    outgoing = {
        account: group
        for account, group in value.groupby("FROM_ADDRESS", sort=False)
    }
    incoming = {
        account: group
        for account, group in value.groupby("TO_ADDRESS", sort=False)
    }

    def window_stats(groups, account, timestamp, seconds, counterparty):
        """Strictly earlier events in [timestamp - seconds, timestamp)."""
        group = groups.get(account)
        if group is None:
            return 0, 0.0, 0

        times = group["TIMESTAMP"].to_numpy()
        lo = np.searchsorted(times, timestamp - seconds, side="left")
        hi = np.searchsorted(times, timestamp, side="left")
        window = group.iloc[lo:hi]

        return (
            len(window),
            float(window["VALUE"].sum()),
            int(window[counterparty].nunique()),
        )

    targets = data.loc[
        data["TIMESTAMP"].ge(reporting_start)
        & data["TIMESTAMP"].lt(reporting_end)
    ].copy()

    features = pd.DataFrame(
        index=pd.Index(
            targets["TRANSACTION_HASH"],
            name="TRANSACTION_HASH",
        )
    )
    metadata = targets.set_index("TRANSACTION_HASH")[
        [
            "transfer_id", "TIMESTAMP",
            "FROM_ADDRESS", "TO_ADDRESS", "VALUE", "is_dust",
        ]
    ].copy()

    features["log_value"] = np.log1p(
        metadata["VALUE"].to_numpy()
    )

    for side, address_column in [
        ("sender", "FROM_ADDRESS"),
        ("receiver", "TO_ADDRESS"),
    ]:
        metadata[f"{side}_known_service"] = metadata[
            address_column
        ].isin(registry.index)

        for field in [
            "service_name", "service_type", "source", "verified_date"
        ]:
            metadata[f"{side}_{field}"] = metadata[
                address_column
            ].map(registry[field])

    metadata["account_class"] = np.where(
        metadata["sender_known_service"]
        | metadata["receiver_known_service"],
        "known_service_involved",
        "unknown_service_status",
    )

    # Treat service-involved transactions as a separate review/model class.
    metadata["ordinary_model_eligible"] = (
        ~metadata["is_dust"]
        & metadata["account_class"].eq("unknown_service_status")
    )

    # Context-completeness flags belong in metadata, not model inputs.
    metadata["past_context_complete"] = (
        metadata["TIMESTAMP"] - max(horizons.values()) >= start
    )
    metadata["future_context_complete"] = (
        metadata["TIMESTAMP"] + max(horizons.values()) < end
    )

    # Trailing activity and distinct counterparties at multiple horizons.
    for label, seconds in horizons.items():
        for side, address_column in [
            ("sender", "FROM_ADDRESS"),
            ("receiver", "TO_ADDRESS"),
        ]:
            addresses = metadata[address_column].to_numpy()
            timestamps = metadata["TIMESTAMP"].to_numpy()

            sent = [
                window_stats(
                    outgoing, account, timestamp, seconds, "TO_ADDRESS"
                )
                for account, timestamp in zip(addresses, timestamps)
            ]
            received = [
                window_stats(
                    incoming, account, timestamp, seconds, "FROM_ADDRESS"
                )
                for account, timestamp in zip(addresses, timestamps)
            ]

            features[f"{side}_sent_n_{label}"] = [
                result[0] for result in sent
            ]
            features[f"{side}_sent_xrp_{label}"] = [
                result[1] for result in sent
            ]
            features[f"{side}_fan_out_{label}"] = [
                result[2] for result in sent
            ]

            features[f"{side}_received_n_{label}"] = [
                result[0] for result in received
            ]
            features[f"{side}_received_xrp_{label}"] = [
                result[1] for result in received
            ]
            features[f"{side}_fan_in_{label}"] = [
                result[2] for result in received
            ]

    # Explicit trailing ten-minute bursts, excluding the current payment.
    for side, address_column, groups, counterparty in [
        ("sender", "FROM_ADDRESS", outgoing, "TO_ADDRESS"),
        ("receiver", "TO_ADDRESS", incoming, "FROM_ADDRESS"),
    ]:
        features[f"{side}_burst_10m_before"] = [
            window_stats(
                groups, account, timestamp, TEN_MINUTES, counterparty
            )[0]
            for account, timestamp in zip(
                metadata[address_column], metadata["TIMESTAMP"]
            )
        ]

    target_ids = metadata["transfer_id"]

    # For the current send, how much is allocated to earlier receipts?
    matched_send = matches.groupby("send_id")["matched_drops"].sum()
    features["sender_observed_receipt_fraction"] = (
        target_ids.map(matched_send).fillna(0).to_numpy()
        / targets["DROPS"].to_numpy()
    )

    for label, seconds in horizons.items():
        recent = matches.loc[matches["delay_seconds"] <= seconds]
        allocated = recent.groupby("send_id")["matched_drops"].sum()

        features[f"sender_recent_receipt_fraction_{label}"] = (
            target_ids.map(allocated).fillna(0).to_numpy()
            / targets["DROPS"].to_numpy()
        )

    # Missing timing is represented explicitly, not as a fabricated delay.
    if not matches.empty:
        weighted = matches.assign(
            weighted_delay=(
                matches["matched_drops"] * matches["delay_seconds"]
            )
        ).groupby("send_id").agg(
            weighted_delay=("weighted_delay", "sum"),
            matched_drops=("matched_drops", "sum"),
        )
        delay = weighted["weighted_delay"] / weighted["matched_drops"] / 60
    else:
        delay = pd.Series(dtype=float)

    metadata["sender_weighted_receipt_age_minutes"] = target_ids.map(delay)
    features["sender_has_matched_receipt"] = (
        metadata["sender_weighted_receipt_age_minutes"].notna()
    )
    features["sender_log_receipt_age_minutes"] = np.log1p(
        metadata["sender_weighted_receipt_age_minutes"].fillna(0)
    )

    if mode == "retrospective":
        # These features require future observation.
        for label, seconds in horizons.items():
            later = matches.loc[matches["delay_seconds"] <= seconds]
            forwarded = later.groupby("receipt_id")["matched_drops"].sum()

            features[f"receiver_forwarded_fraction_{label}"] = (
                target_ids.map(forwarded).fillna(0).to_numpy()
                / targets["DROPS"].to_numpy()
            )

            # At least 90% of this receipt was allocated to later sends.
            features[f"receiver_forwarded_90pct_{label}"] = (
                features[f"receiver_forwarded_fraction_{label}"] >= 0.9
            )

        first_forward = matches.groupby("receipt_id")["delay_seconds"].min()
        metadata["receiver_minutes_to_first_matched_send"] = (
            target_ids.map(first_forward) / 60
        )

        features["receiver_has_observed_forward"] = metadata[
            "receiver_minutes_to_first_matched_send"
        ].notna()
        features["receiver_log_minutes_to_forward"] = np.log1p(
            metadata["receiver_minutes_to_first_matched_send"].fillna(0)
        )

    features = features.astype(float)

    metadata["model_eligible"] = (
        metadata["ordinary_model_eligible"]
        & metadata["past_context_complete"]
    )
    if mode == "retrospective":
        metadata["model_eligible"] &= metadata["future_context_complete"]

    audit.update({
        "feature_mode": mode,
        "dust_threshold_xrp": dust_xrp,
        "context_payment_count": len(data),
        "context_dust_count": int(data["is_dust"].sum()),
        "context_dust_xrp": float(
            data.loc[data["is_dust"], "VALUE"].sum()
        ),
        "report_payment_count": len(targets),
        "ordinary_model_eligible_count": int(
            metadata["model_eligible"].sum()
        ),
        "observation_boundaries_inferred": (
            observed_start is None or observed_end is None
        ),
    })

    return {
        "features": features,
        "metadata": metadata,
        "matches": matches,
        "transactions": data,
        "dust": data.loc[data["is_dust"]].copy(),
        "audit": audit,
    }


def find_linked_routes(
    transactions,
    matches,
    *,
    max_hops=4,
    max_minutes=7 * 24 * 60,
    min_link_xrp=1.0,
    max_search_states=2_000_000,
    max_routes=100_000,
):
    """
    Produce chronological FIFO-linked evidence routes.

    Finds:
      - cycles returning to the starting address;
      - open chains of at least two hops, which can be reviewed for peeling.

    An open chain is not itself proof of a peel chain.
    Amounts are pairwise-link upper bounds, not proven circulated value.
    Overlapping routes must not be summed.

    This is retrospective evidence and is not added to causal model inputs.
    """
    if max_hops < 2 or max_minutes <= 0 or min_link_xrp <= 0:
        raise ValueError("Invalid route-search limits.")

    transfers = transactions.set_index("transfer_id").to_dict("index")
    successors = defaultdict(list)

    for row in matches.itertuples(index=False):
        if row.matched_drops >= min_link_xrp * 1_000_000:
            successors[int(row.receipt_id)].append(
                (int(row.send_id), int(row.matched_drops))
            )

    rows = []
    states = 0

    def walk(start_id, path, accounts, capacity):
        nonlocal states
        states += 1

        if states > max_search_states:
            raise RuntimeError(
                "Route search limit exceeded. Reduce hops/time horizon "
                "or partition the analysis; no complete result is available."
            )

        if len(path) >= max_hops:
            return

        first = transfers[start_id]
        current = transfers[path[-1]]

        for next_id, allocation in successors.get(path[-1], []):
            nxt = transfers[next_id]

            if nxt["FROM_ADDRESS"] != current["TO_ADDRESS"]:
                continue
            if nxt["TIMESTAMP"] <= current["TIMESTAMP"]:
                continue

            duration = (nxt["TIMESTAMP"] - first["TIMESTAMP"]) / 60
            if duration > max_minutes:
                continue

            closes_cycle = nxt["TO_ADDRESS"] == first["FROM_ADDRESS"]
            if not closes_cycle and nxt["TO_ADDRESS"] in accounts:
                continue

            next_path = path + [next_id]
            next_accounts = accounts + [nxt["TO_ADDRESS"]]
            next_capacity = min(capacity, allocation)

            if len(rows) >= max_routes:
                raise RuntimeError(
                    "Route output limit exceeded. Reduce hops/time horizon; "
                    "no complete result is available."
                )

            rows.append({
                "route_type": (
                    "chronological_cycle"
                    if closes_cycle
                    else "open_chain_for_review"
                ),
                "transfer_ids": json.dumps(next_path),
                "transaction_hashes": json.dumps([
                    transfers[i]["TRANSACTION_HASH"] for i in next_path
                ]),
                "accounts": json.dumps(next_accounts),
                "hops": len(next_path),
                "start_timestamp": first["TIMESTAMP"],
                "end_timestamp": nxt["TIMESTAMP"],
                "duration_minutes": duration,
                "fifo_link_upper_bound_xrp": next_capacity / 1_000_000,
            })

            if not closes_cycle:
                walk(
                    start_id, next_path, next_accounts, next_capacity
                )

    for transfer_id, transfer in transfers.items():
        if transfer["DROPS"] < min_link_xrp * 1_000_000:
            continue
        if transfer_id not in successors:
            continue

        walk(
            transfer_id,
            [transfer_id],
            [transfer["FROM_ADDRESS"], transfer["TO_ADDRESS"]],
            int(transfer["DROPS"]),
        )

    return pd.DataFrame(
        rows,
        columns=[
            "route_type", "transfer_ids", "transaction_hashes",
            "accounts", "hops", "start_timestamp", "end_timestamp",
            "duration_minutes", "fifo_link_upper_bound_xrp",
        ],
    )


# ---------------------------------------------------------------------
# Usage
# ---------------------------------------------------------------------

# Populate from independently verified sources.
# Use an appropriate historical snapshot for historical evaluation.
services = pd.DataFrame(
    columns=[
        "address",
        "service_name",
        "service_type",
        "source",
        "verified_date",
    ]
)

# Supply a broader context dataset here if available.
# report_start/report_end select target transactions without cutting away
# earlier/later context needed for matching and feature windows.
result = make_features(
    train,
    services=services,
    dust_xrp=1.0,       # Example threshold: check sensitivity.
    mode="causal",     # "retrospective" adds later forwarding features.
    horizons=HORIZONS,
    report_start=None,
    report_end=None,
    observed_start=None,  # Prefer actual ingestion coverage boundaries.
    observed_end=None,
)

train_features = result["features"]
metadata = result["metadata"]

# Known-service involvement, dust and incomplete windows are excluded
# from this ordinary-account model input, but remain available for review.
X = train_features.loc[metadata["model_eligible"]].copy()

print("Audit:")
print(pd.Series(result["audit"]).to_string())

print("\nFeature summary:")
print(X.describe().T[["mean", "max"]])

if X.empty:
    print(
        "\nNo model-eligible transactions remain. Supply more context "
        "or deliberately choose shorter horizons."
    )

# Optional retrospective route evidence. Can be expensive on large graphs.
# routes = find_linked_routes(
#     result["transactions"],
#     result["matches"],
#     max_hops=4,
#     max_minutes=7 * 24 * 60,
#     min_link_xrp=1.0,
# )

# ---------------------------------------------------------------------
# Labels and evaluation
# ---------------------------------------------------------------------

# Do not automatically treat train.is_suspicious as ground truth.
# Leave False if labels were generated from these or similar rules.
LABELS_ARE_INDEPENDENTLY_ASSESSED = False
y = None

if LABELS_ARE_INDEPENDENTLY_ASSESSED:
    if "is_suspicious" not in train.columns:
        raise ValueError("No independently assessed label column supplied.")

    labelled = train[["TRANSACTION_HASH", "is_suspicious"]].copy()

    if (
        labelled.groupby("TRANSACTION_HASH")["is_suspicious"]
        .nunique(dropna=False)
        .gt(1)
        .any()
    ):
        raise ValueError("Conflicting labels for a transaction.")

    labels = (
        labelled.drop_duplicates("TRANSACTION_HASH")
        .set_index("TRANSACTION_HASH")["is_suspicious"]
        .reindex(X.index)
    )

    if labels.isna().any():
        raise ValueError("Some model-eligible transactions lack labels.")

    if not labels.isin([0, 1, False, True]).all():
        raise ValueError("Labels must be binary.")

    y = labels.astype(int).to_numpy()

# Evaluation requirements:
# - Split chronologically before fitting preprocessing or a model.
# - Construct each split from data permitted at its decision cutoff.
# - Retrospective features require delayed decisions and mature windows.
# - Purge overlapping label/feature intervals across split boundaries.
# - Where possible, group related accounts to prevent identity leakage.
# - Keep the final test set untouched during threshold selection.
# - With rule-generated labels, report rule agreement, not independent
#   detection accuracy.
#
# Reporting:
# - Use "flagged for review", not "criminal".
# - Present matched hashes, timestamps, amounts and service context.
# - Pseudonymise addresses in public slides.
# - Known-service status is context, not evidence of legitimacy.
# - FIFO allocation is an assumption, not proof of fund provenance.
# - A missing observation is not proof that no transfer occurred.

NameError: name 'train' is not defined